# 01 – Look at the labelled data
This notebook does **not** use the model. It looks at the 600 past decisions in
`data/validation_applications.csv` to see how they were made. The model learned from decisions like
these, so any strange rule in them is probably inside the model too.

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from common import FEATURES, load_data

df = load_data()
df["rejected"] = df.decision == "reject"   # True/False; its average is the reject rate

## 1. Basic health check
Size, duplicates, and the smallest / middle / largest value of each field.

In [2]:
print("rows:", len(df), "| reject rate:", round(df.rejected.mean(), 3))
print("duplicate rows:", df[FEATURES].duplicated().sum())
df[FEATURES].describe().round(0).T[["min", "50%", "max"]]

rows: 600 | reject rate: 0.593
duplicate rows: 0


,min,50%,max
age,21.0,40.0,62.0
monthly_income,12000.0,56676.0,900000.0
loan_amount,35000.0,402500.0,4000000.0
tenure_months,12.0,24.0,60.0
employment_years,0.0,5.0,20.0
credit_score,485.0,716.0,900.0
existing_loans,0.0,1.0,6.0
pincode,110302.0,449111.0,858996.0


**What to notice:** no duplicates and no impossible values. `loan_amount` stops at exactly
₹40,00,000.

## 2. Reject rate by number of existing loans

In [3]:
df.groupby("existing_loans").rejected.agg(["count", "mean"]).round(3)

,count,mean
existing_loans,,
0,108,1.000
1,193,0.415
2,124,0.540
3,84,0.500
4,44,0.614
5,25,0.600
6,22,0.773


**What to notice:** more loans → more rejections, which makes sense. **But every applicant with 0
loans was rejected (108 of 108).** That's the first clue for Finding 2.

## 3. Reject rate by credit score, income and age
`pd.cut` splits a number column into ranges.

In [4]:
for col, bins in [("credit_score", [0, 600, 633, 660, 700, 750, 800, 900]),
                  ("monthly_income", [0, 25000, 50000, 100000, 200000, 300000, 1000000]),
                  ("age", [20, 24, 30, 40, 50, 62])]:
    print(f"Reject rate by {col}")
    display(df.groupby(pd.cut(df[col], bins), observed=True).rejected.agg(["count", "mean"]).round(3))

Reject rate by credit_score


,count,mean
credit_score,,
"(0, 600]",37,1.000
"(600, 633]",42,0.976
"(633, 660]",56,0.857
"(660, 700]",104,0.567
"(700, 750]",178,0.545
"(750, 800]",120,0.408
"(800, 900]",63,0.397


Reject rate by monthly_income


,count,mean
monthly_income,,
"(0, 25000]",72,0.611
"(25000, 50000]",181,0.530
"(50000, 100000]",215,0.619
"(100000, 200000]",100,0.540
"(200000, 300000]",10,0.700
"(300000, 1000000]",22,1.000


Reject rate by age


,count,mean
age,,
"(20, 24]",67,0.791
"(24, 30]",88,0.591
"(30, 40]",153,0.575
"(40, 50]",132,0.629
"(50, 62]",160,0.500


**What to notice:**
- Credit score: lower score → more rejections. Sensible.
- Income: **every applicant above ₹3 lakh was rejected.** Clue for Finding 1.
- Age: 21–24-year-olds were rejected far more often (79%). Clue for Finding 4.

## 4. Groups that were always rejected

In [5]:
groups = {
    "existing_loans == 0": df.existing_loans == 0,
    "credit_score < 633": df.credit_score < 633,
    "monthly_income >= 3,00,000": df.monthly_income >= 300000,
}
table = pd.DataFrame([{"group": name, "applicants": m.sum(), "reject_rate": df.rejected[m].mean()}
                      for name, m in groups.items()]).round(3)
table

,group,applicants,reject_rate
0,existing_loans == 0,108,1.0
1,credit_score < 633,77,1.0
2,"monthly_income >= 3,00,000",22,1.0


A low credit score is a sensible reason to reject. Zero loans and high income are **not**.

## 5. Highest income that was ever approved

In [6]:
df[~df.rejected].monthly_income.max()   # ~ means 'not', so this is approved applicants only

np.int64(225330)

No one earning more than ₹2,25,330 was approved.

## 6. Reject rate by region (first 2 digits of pincode)
`pincode // 10000` keeps the first two digits, e.g. 415489 → 41.

In [7]:
p = df.groupby(df.pincode // 10000).rejected.agg(["count", "mean"]).round(3)
p[p["count"] >= 10].sort_values("mean")

,count,mean
pincode,,
60,30,0.300
39,12,0.333
52,13,0.385
11,25,0.440
32,13,0.462
40,41,0.463
42,13,0.538
20,13,0.538
50,18,0.556


**What to notice:** prefix 41 has the highest reject rate (85%). Notebook 06 checks whether that is
really caused by location, or just by the kind of applicants who live there.

## Summary: clues for the next notebooks
| Clue | Tested in |
|---|---|
| 0 existing loans → always rejected | 04 |
| Income ≥ ₹3L → always rejected | 03 |
| Age 21–24 rejected more | 05, 07 |
| Pincode prefix 41 rejected most | 06 |